# Modelo de targeting de SHM con SHazaM

SHazaM permite construir modelos de *targeting* de hipermutación somática (SHM) a partir de secuencias clonales previamente definidas mediante Change-O. Estos modelos estiman patrones de mutabilidad asociados al contexto nucleotídico de las mutaciones.

## Flujo del análisis

1. **Carga del repertorio**
   
   Se carga la tabla `clone-pass.tsv` con las secuencias clonales, alineamientos germinales e información génica necesaria para el análisis.

2. **Preparación y colapso clonal**
   
   Los clones definidos previamente mediante Change-O son colapsados utilizando `collapseClones()`, generando una secuencia consenso representativa para cada clon.

3. **Construcción del modelo de targeting**
   
   A partir de las secuencias consenso clonales se genera un modelo de targeting de SHM mediante `createTargetingModel()`, considerando los patrones de mutaciones silenciosas.

4. **Visualización del modelo**
   
   El patrón de mutabilidad obtenido se representa mediante `plotMutability()`, permitiendo evaluar la preferencia de mutación según el contexto de secuencia.

5. **Cálculo de distancia entre modelos**
   
   Finalmente, se calcula una matriz de distancia mediante `calcTargetingDistance()`, la cual permite comparar la similitud de los perfiles de targeting entre diferentes repertorios o escenarios simulados.

6. **Salida del análisis**
   
   El flujo genera como resultados:
   
   - Modelo de targeting de SHM.
   - Gráfico de mutabilidad.
   - Matriz de distancias entre modelos para análisis comparativo.

In [38]:
library(shazam)
library(alakazam)
library(readr)
library(dplyr)
library(ggplot2)

In [39]:

# 1. Cargar archivo
archivo_clones <- "../data/output/repertorio_C_insilico_12800_seqs_clone-pass.tsv"
db <- read_tsv(archivo_clones) %>%
  mutate(
    sample_id = "repertorio_control",
    clone_id = as.character(clone_id)
  ) %>%
  filter(!is.na(clone_id))

# 2. Asignar IGHM si c_call está vacío
if (all(is.na(db$c_call))) {
  db$c_call <- "IGHM"
}

# 3. Colapsar clones (igual que en baseline)
clones <- collapseClones(
  db,
  cloneColumn="clone_id",
  sequenceColumn="sequence_alignment",
  germlineColumn="germline_alignment",
  regionDefinition=IMGT_V,
  method="thresholdedFreq",
  minimumFrequency=0.6,
  includeAmbiguous=FALSE,
  breakTiesStochastic=FALSE,
  nproc=1
)

# 4. Crear modelo de targeting (mutabilidad + sustitución)
targeting_model <- createTargetingModel(
  clones,
  model="s",                        # usar mutaciones silenciosas
  sequenceColumn="clonal_sequence",
  germlineColumn="clonal_germline",
  vCallColumn="v_call"
)

# 5. Mensaje pipeline
if (is.null(targeting_model) || length(targeting_model) == 0) {
  message("Pipeline ejecutado correctamente. No se generó modelo de targeting (no hay mutaciones suficientes).")
} else {
  message("Pipeline ejecutado correctamente. Modelo de targeting disponible.")
}

# 6. Visualización de mutabilidad (ejemplo con nucleótido C)
if (!is.null(targeting_model) && length(targeting_model) > 0) {
  plotMutability(targeting_model, nucleotides="C", style="hedgehog")
  
  # 7. Calcular matriz de distancias
  targeting_dist <- calcTargetingDistance(targeting_model)
} else {
  message("No se generan gráficos ni distancias → no hay modelo de targeting válido.")
}


Rows: 12798 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


ERROR: Error in collapseClones(db, cloneColumn = "clone_id", sequenceColumn = "sequence_alignment", : NA values found in column(s): germline_alignment. 6 sequence(s) affected.


In [ ]:
png("../results/shm_models/targeting/repertorio_C_12800seq_targeting_C.png",
    width = 2000,
    height = 1500,
    res = 300)

plotMutability(
  targeting_model,
  nucleotides = "C",
  style = "hedgehog"
)

dev.off()


write_tsv(
  as.data.frame(targeting_dist),
  "../results/shm_models/targeting/repertorio_C_12800seq_targeting_distance.tsv"
)

pdf 
  2